# ⚛️ qAIR-vNext **v46** — the diagnostics release

Runs on a **Colab GPU runtime**, with code pulled from GitHub and every
artifact written to **Google Drive**. Nothing trains locally any more.

    Drive (persistent)          Colab VM (ephemeral)
    ├── cache/   embeddings  ←  git clone <repo>
    ├── ckpt/    checkpoints    pip install -r requirements.txt
    └── exports/ figures

Capped at **80 train / 20 validation** on purpose. This run answers one
question — *does the pipeline execute end to end without breaking?* — and
it answers nothing else. At 80 examples against ~1.05M parameters the
model memorizes the training set in a couple of epochs; every accuracy
below is noise with a standard error near **10 points**. Do not read a
result into it.

---

## What v46 changes, and why

v46 changes **no parameter, no module, no hyperparameter, no loss term and
no cache field**. v45.1 checkpoints load into it unchanged and
`CACHE_VERSION` stays at **45**. It adds measurement — because the v45.1
ARC-Challenge run exposed a diagnostic that could not fail.

`destructive_fraction` read exactly **1.0000** every epoch of that run. It
was not reporting learning. With `K = 2N` matched support/attack pairs and
near-equal amplitudes it is a **step function of one scalar**,
`full_model.polarity_phase`:

| `polarity_phase` | 0.0 | 1.6 | 2.0 | 2.4 | 2.8 | 3.0 | π |
|---|---|---|---|---|---|---|---|
| `destructive_fraction` | 0.000 | 0.000 | 0.004 | 0.494 | 0.994 | 1.000 | 1.000 |

That scalar is **initialized at π** — already past the step — so the metric
reads 1.0 for an untrained model and stays there. The existing
`[INTERFERENCE WARNING]` guard only fires *below* 0.01, so this regime
never tripped anything.

It cannot report the **sign** of the effect either. Ranking options by the
coherent sum versus by the decohered mixture over identical amplitudes and
energies: with no support/attack amplitude asymmetry, `polarity_phase = π`
puts the coherent ranking well *below* the mixture; with the asymmetry the
mechanism needs, well *above*. `destructive_fraction` is ~1 in both.

| added | where | what it decides |
|---|---|---|
| `coherent_acc` vs `classical_acc` | §10 epoch report | **the decisive number.** Same amplitudes, same energies, one forward pass, ranked coherently vs decohered. `coherent_acc ≤ classical_acc` means the complex amplitudes are not earning their place, whatever the interference diagnostics say. Excludes the LLM prior, so it separates the architecture from the prior |
| `coherent_contrast` | §10 epoch report | `(max−min)/mean` over options. **Not** a good-news number — it *rises* under cancellation, because the relative spread of a small residual is amplified. Read it against `ece` |
| `polarity_phase` itself | §10 epoch report (verbose) | the cause. Logging the effect without the cause is what let the v45.1 run go seven epochs |
| `[INTERFERENCE SATURATED]` | `training/train.py` | mirror of the existing low-side guard — fires on `destructive_fraction > 0.99` **and** `coherent_acc ≤ classical_acc`. Both directions now have an alarm |
| `--mode asymmetry` | **§6c, before training** | cache-level, training-free. Tests the premise the whole design rests on: that the attack on the *correct* option is weaker than the attacks on the distractors. If it is not, there is nothing for the collapse to discriminate on |

**Nothing here is tuned.** Changing a model variable in the same commit as
the instrument that measures it would make this run uninterpretable.
Fixing `polarity_phase` is the *next* experiment, and it needs this
instrumentation in place first to be interpretable.

The v45 architecture changes this builds on — support/attack evidence,
complex amplitudes, orthogonal mixing, the width cut — are unchanged; see
`README.md` and `notebooks/qAIR_v45.ipynb` for that table.

<a id="1"></a>
## 1. Runtime check

Two independent things run on "GPU" in this notebook and it's worth
naming them separately: the **classical layers** (embeddings, reasoner,
selector) run on whatever `torch` puts on CUDA, and the **quantum
circuit** runs on whichever PennyLane simulator attaches --
`lightning.gpu` (NVIDIA cuStateVec) if it's installed and a GPU is
present, else `lightning.qubit` on CPU. §4 installs both; §8 prints which
one actually attached.

**Runtime → Change runtime type → T4 GPU** before running anything.

In [ ]:
import subprocess, sys

print(sys.version)

try:
    print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total",
                                   "--format=csv,noheader"]).decode().strip())
except Exception:
    print("\n[NO GPU] This will still run, but cache building will be slow.\n"
          "Runtime -> Change runtime type -> T4 GPU.")

<a id="2"></a>
## 2. Mount Drive

The Colab VM is wiped when the session ends; Drive is not. Cache and
checkpoints live there so a disconnect costs you nothing — the cache
build autosaves every 50 samples and resumes, and the trainer writes a
`_latest.pt` every epoch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = "/content/drive/MyDrive/qAIR_V45"

CACHE_DIR = f"{BASE}/cache"
CKPT_DIR = f"{BASE}/ckpt"
EXPORT_DIR = f"{BASE}/exports"

for p in (CACHE_DIR, CKPT_DIR, EXPORT_DIR):
    os.makedirs(p, exist_ok=True)

print("BASE:", BASE)
!ls -la "{BASE}"

<a id="3"></a>
## 3. Clone the repo

Source of truth is GitHub, not Drive - a stale copy of the code sitting
next to a fresh cache is how two "identical" runs stop being comparable.

> The repo was renamed **qAIR-CSE499A -> qAIR-CSE499B**. The old URL still
> redirects, so either works, but the one below is canonical. If a clone
> 404s, check git remote -v in your local checkout.


In [ ]:
REPO_URL = "https://github.com/Thorfast191/qAIR-CSE499B.git"
BRANCH = "main"

!rm -rf /content/qAIR
!git clone --branch {BRANCH} {REPO_URL} /content/qAIR

%cd /content/qAIR
!git log --oneline -3

<a id="4"></a>
## 4. Install dependencies

Colab ships TensorFlow/Keras preinstalled, which shadows `transformers`'
backend detection — remove them first.

`requirements.txt` deliberately does not pin `torch`; Colab's preinstalled
build is CUDA-correct for its own runtime, so leave it alone.

`pennylane-lightning[gpu]` is installed separately, not from
`requirements.txt` — it needs an NVIDIA GPU + cuStateVec, which the base
requirements file deliberately excludes so `pip install -r
requirements.txt` doesn't break on a CPU-only machine. Installing it here
is what lets `models/quantum_layer.py`'s circuit run on the GPU instead of
falling back to the CPU simulator (`config.QUANTUM_DEVICE = "auto"` picks
whichever attached — §8 prints which one).

In [ ]:
!pip uninstall -y -q tensorflow tensorflow-cpu keras tf-keras
!pip install -q -r requirements.txt

# NVIDIA cuStateVec-backed simulator for the quantum circuit. Only makes
# sense with a GPU runtime attached (§1) -- on CPU this install is
# harmless but useless, and QUANTUM_DEVICE="auto" falls back to
# lightning.qubit regardless of whether this succeeded. `!pip` failing
# does not stop the notebook, so no error handling is needed here --
# just attempt it and let §8 report which device actually attached.
!pip install -q "pennylane-lightning[gpu]==0.42.0"

### 4b. Imports, seed, device

`set_seed` existed in this project for its entire history and was **never
called from any entry point**, so no run was ever reproducible —
`collate_fn` draws a fresh `randperm` per batch, dropout is live, and
generation samples. It is wired into every entry point now, and here.

If the import cell fails with a `numpy`/`torch` ABI error, use
**Runtime → Restart session** and run from §2 again (Drive stays
mounted); the pip install above replaced a package Colab had already
imported.

In [ ]:
from functools import partial

import torch
from torch.utils.data import DataLoader

from config import (
    ARC_CONFIG, ARC_SPLIT_SIZES, BATCH_SIZE, PATIENCE, PERSISTENT_STEPS,
    N_QUBITS, MODEL_DIM, WEIGHT_DECAY, SEED, GEN_MODES, PHASE_MODE,
    PHASE_SOURCE, MIXING, USE_LLM_PRIOR, QUANTUM_DEVICE,
    EMBEDDING_DIM as DIM, resolve_device,
)
from training.seed import set_seed, seeded_generator
from training.dataset import QAIRDataset, collate_fn
from training.train import Trainer
from training.checkpoint import load_or_resume
from training.evaluate import evaluate
from models.full_model import QAIRvNext

set_seed(SEED)

device = resolve_device()

print(f"torch device (classical layers): {device}")
print(f"quantum device preference      : {QUANTUM_DEVICE}   "
      f"(resolved when the model is built, in §8)")
print(f"benchmark     : {ARC_CONFIG}")
print(f"embedding dim : {DIM}   model dim: {MODEL_DIM}")
print(f"gen modes     : {GEN_MODES}")
print(f"phase         : mode={PHASE_MODE} source={PHASE_SOURCE}  mixing={MIXING}")

<a id="5"></a>
## 5. Run configuration

`TRAIN_SAMPLES` / `VAL_SAMPLES` are the important knobs.

> **These caps must stay consistent everywhere.** A cache built with
> `max_samples` is stored `complete=False` *by design*, so a later full
> build can resume and finish the split. The consequence: passing
> `max_samples=None` against a capped cache does **not** read 80
> samples — it starts generating the remaining ~1,039.

In [ ]:
RUN_NAME = "qair_v46"

# To re-measure the STOPPED v45.1 ARC-Challenge run under this
# instrumentation instead of starting a fresh one, set RUN_NAME to
# "qair_v45.1" and both caps below to None. v46 adds no parameter and no
# module, so that checkpoint loads unchanged and load_or_resume picks the
# run up at the epoch it stopped on -- which is the only way to read
# coherent_acc vs classical_acc against the numbers already logged.

TRAIN_SAMPLES = 80      # cap -- must match how the cache was built
VAL_SAMPLES = 20        # None = full split (ARC_SPLIT_SIZES[ARC_CONFIG])

EPOCHS = 8              # 80 samples converges (i.e. memorizes) fast

persistent_steps = PERSISTENT_STEPS
n_qubits = N_QUBITS

# The validator-feedback pass runs the reasoner AND the quantum layer
# twice per forward. The circuit is still the dominant per-epoch cost even
# on lightning.gpu (it runs once per hypothesis, per sample, per batch),
# so turning this off roughly halves per-epoch time while iterating.
VALIDATOR_FEEDBACK = True

# VAL_SAMPLES=None means "no cap", not "zero" -- the dataset itself isn't
# loaded until §6/§7, so the SE preview below has to fall back to the
# known split size instead of dividing by None.
_val_n_preview = VAL_SAMPLES or ARC_SPLIT_SIZES[ARC_CONFIG]["validation"]

print(f"{RUN_NAME}: {TRAIN_SAMPLES or 'all'} train / {VAL_SAMPLES or 'all'} val, "
      f"{EPOCHS} epochs, feedback={VALIDATOR_FEEDBACK}")
print(f"\nSE at n={_val_n_preview} is ~{(0.25*0.75/_val_n_preview)**0.5:.3f} "
      f"-- i.e. +/-{100*(0.25*0.75/_val_n_preview)**0.5:.0f} points at one sigma.")
print("Nothing measured in this notebook is a result. It is a plumbing test.")

<a id="6"></a>
## 6. Build the cache

The slow one-time step. For every question the LLM writes **two**
hypotheses per option — one supporting, one attacking — and then scores
each option's log-likelihood directly. All of it is embedded and cached
to Drive.

At these caps that is 80 × 4 × 2 = ~640 completions for train and ~160
for validation: a few minutes on a T4. The full ARC-Challenge splits
(1,119 / 299) are roughly 15× that.

Resumable and atomic: autosaves every 50 samples with the raw dataset
index it reached, so a disconnect costs at most 50 samples. Rerun this
cell to resume.

In [ ]:
train_ds = QAIRDataset(split="train", max_samples=TRAIN_SAMPLES,
                       cache_dir=CACHE_DIR, arc_config=ARC_CONFIG)

val_ds = QAIRDataset(split="validation", max_samples=VAL_SAMPLES,
                     cache_dir=CACHE_DIR, arc_config=ARC_CONFIG)

print(f"\nTrain samples: {len(train_ds)}")
print(f"Val samples:   {len(val_ds)}")

assert len(train_ds) > 0 and len(val_ds) > 0, "Empty dataset -- cache build failed."

### 6b. ⭐ The cache-quality gate

**Read the `[CACHE QUALITY]` block printed above before going further.**

v44 reported one number here — the template-fallback rate — and it read
0% on a cache that was nevertheless useless. A fluent hypothesis that
justifies every option is uninformative in exactly the same way a
template is; it just doesn't look it.

So the gate now measures **discriminative signal** directly, as zero-shot
accuracies that need no training:

| statistic | what it asks |
|---|---|
| support agreement | does the supporting hypothesis for the *correct* option match it best? |
| attack disagreement | does the *correct* option attract the weakest objection? |
| support − attack margin | is the support/attack asymmetry informative? |
| LLM option likelihood | what does the generator think, unaided? |

If none of the first three beats chance, **stop**. The prompts produced
symmetric evidence, and no architecture trained on this cache can do
better than option priors. That is a generation problem; fix
`models/generator.py` and rebuild. Tuning the model cannot recover a
signal that was never generated — that is the mistake this project spent
a year making.

In [ ]:
s = train_ds[0]

n_opt = len(s["options"])

print("Question:", s["question"])
print()

for i, opt in enumerate(s["options"]):
    mark = "*" if i == s["y"] else " "
    print(f" {mark}[{i}] {opt}    (llm logp {s['llm_logprob'][i]:+.3f})")
    for m, mode in enumerate(GEN_MODES):
        j = m * n_opt + i
        fb = "  [FALLBACK]" if bool(s["is_fallback"][j]) else ""
        print(f"       {mode:<8s}: {s['hypotheses'][j]}{fb}")

print()
print("H:", tuple(s["H"].shape), " O:", tuple(s["O"].shape),
      " Q:", tuple(s["Q"].shape))
print("polarity  :", s["polarity"].tolist())
print("hyp_option:", s["hyp_option"].tolist())

In [ ]:
# Re-print the gate for the validation split too -- this is the split
# every number below is measured on.
val_ds.report_quality("validation")

In [ ]:
# 6c. Reference baselines and the asymmetry check -- BEFORE training.
#
# v46 adds --mode asymmetry to this suite, and --mode all now includes it.
# It tests the premise the coherent collapse rests on: that the attacking
# hypothesis for the CORRECT option is weaker than the attacks on the
# distractors. With the hypothesis phases at 0 and pi the coherent sum
# reduces to (sum_support w - sum_attack w)^2, so an attack gap
# indistinguishable from zero means there is nothing to discriminate on --
# a GENERATION problem. Change the prompts in models/generator.py and
# rebuild; training against a dead cache is the mistake this project is a
# case study in. Read it now, not after eight epochs.
#
# The caps are threaded through deliberately. A cache built with
# max_samples is stored complete=False by design, so running this without
# them does not read 80 samples -- it starts GENERATING the rest of the
# split with the LLM, here, in this cell.
cmd = [
    sys.executable, "-m", "evaluation.baselines", "--mode", "all",
    "--split", "validation",
    "--cache-dir", CACHE_DIR,
    "--arc-config", ARC_CONFIG,
]

if TRAIN_SAMPLES is not None:
    cmd += ["--train-samples", str(TRAIN_SAMPLES)]

if VAL_SAMPLES is not None:
    cmd += ["--val-samples", str(VAL_SAMPLES)]

print(" ".join(cmd), "\n")

subprocess.run(cmd, check=True)

<a id="7"></a>
## 7. DataLoaders

`collate_fn` zero-pads ragged option/hypothesis counts (ARC-Challenge has
3- and 5-option questions) and emits `H_mask`/`O_mask`. Those masks were
computed and then **never passed to the model** before v44, so padded
options took part in every reduction and could be returned as the
prediction.

v45 threads three more tensors through: `polarity` (+1 support / −1
attack), `align` (which option each hypothesis argues about), and
`llm_logprob`. Option shuffling permutes all of them together — the
assertion below is what proves it.

In [ ]:
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    generator=seeded_generator(SEED),      # batch ORDER reproducible too
    collate_fn=partial(collate_fn, shuffle_options=True),
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=partial(collate_fn, shuffle_options=False),
)

batch = next(iter(train_loader))

for k, v in batch.items():
    print(f"  {k:<12s} {tuple(v.shape)}")

# Every real hypothesis must point at exactly one real option after the
# option permutation, or the whole support/attack design is misaligned.
rows = batch["align"].sum(-1)
assert bool(((rows == 1.0) | ~batch["H_mask"]).all()), "alignment broken by collate"
print("\nalignment survives option shuffling: OK")

<a id="8"></a>
## 8. Model

```
Q,H,O ─▶ in_proj (384→128)
          │
          ▼
   PersistentReasoner ──▶ Quantum layer ──▶ Validator ──┐
   (orthogonal mixing)    (state,energy,     │          │ potential
                           PHASE)            ▼          │ (closed loop)
                                    EnergyAnswerSelector │
                                      (E_kn, θ_kn)       │
                                             │           │
                                    EnergyFusion ──▶ CollapseController
                                             │           │ (temperature)
                                             ▼           ▼
                                        CoherentCollapse
                                        P(n) ∝ |Σₖ cₖ vₖₙ|²
                                             │
                                             ▼
                                     + LLM log-prior (ablatable)
```

The "Quantum layer" box runs on whichever PennyLane device attached under
`QUANTUM_DEVICE="auto"` — `lightning.gpu` (cuStateVec) if §4's install
succeeded and a GPU is present, else `lightning.qubit` on CPU. Either way
it's the same circuit computing the same function; only the hardware
differs. `verbose=True` below prints which one it is.

In [ ]:
model = QAIRvNext(
    dim=DIM,
    model_dim=MODEL_DIM,
    use_quantum=True,
    use_validator=True,
    persistent_steps=persistent_steps,
    n_qubits=n_qubits,
    use_question=True,          # worth +12.6 pts on a probe; absent until v44
    validator_feedback=VALIDATOR_FEEDBACK,
    phase_mode=PHASE_MODE,
    phase_source=PHASE_SOURCE,
    mixing=MIXING,
    use_llm_prior=USE_LLM_PRIOR,
    use_attack=True,
    quantum_device=QUANTUM_DEVICE,
    verbose=True,
).to(device)

total = sum(p.numel() for p in model.parameters())
circuit = sum(p.numel() for p in model.quantum.quantum.parameters())

print(f"\ncircuit device     : {model.quantum.device_name}   "
      f"({'GPU -- cuStateVec' if model.quantum.device_name == 'lightning.gpu' else 'CPU simulator'})")
print(f"parameters         : {total:,}   (circuit weights: {circuit:,})")
print(f"params / example   : {total / max(len(train_ds), 1):,.0f}")
print(f"                     v44 was 13,223 -- and its best accuracy was at "
      f"epoch 1,\n                     declining after while train loss fell 54%.")

<a id="9"></a>
## 9. Sanity checks — *before* spending time training

Each check targets a specific defect a previous version actually shipped.
Running them first means a broken build fails in seconds rather than
after an hour.

In [ ]:
import torch.nn.functional as F

ok = True

def show(tag, good, detail=""):
    global ok
    ok &= bool(good)
    print(f"  [{'PASS' if good else 'FAIL'}] {tag}   {detail}")

print("SANITY CHECKS")

# ---- F1: does the circuit respond to its input at all? ----------------
# v43 shipped Hadamard + AngleEmbedding(rotation="X"). Hadamard prepares
# |+>, the +1 eigenstate of Pauli-X, and RX on an X-eigenstate is a global
# phase. Output std across distinct inputs was 0.000e+00 -- the layer was
# a learned CONSTANT and every quantum ablation before the fix measured
# nothing at all.
if model.backend == "quantum":
    with torch.no_grad():
        a = model.quantum(torch.randn(1, 4, MODEL_DIM, device=device) * 0.05)
        b = model.quantum(torch.randn(1, 4, MODEL_DIM, device=device) * 0.05)
    show("circuit state responds to input", (a[0]-b[0]).abs().max() > 1e-6)
    show("circuit PHASE responds to input", (a[2]-b[2]).abs().max() > 1e-6,
         f"phase spread {a[2].std().item():.4f}")

# ---- F2: is the hypothesis mixing really orthogonal? -----------------
# The v44 reasoner contracted to a constant fixed point: after 5 steps
# every hypothesis was the same vector (pairwise cos 1.000000) and that
# vector was independent of the input.
H = torch.randn(2, 8, MODEL_DIM, device=device)
_, U = model.reasoner._orthogonal_mix(
    H, F.normalize(H, dim=-1), None, torch.sigmoid(model.reasoner.dt))
I = torch.eye(8, device=device).expand(2, 8, 8)
show("mixing is orthogonal (U Uᵀ = I)",
     (U @ U.transpose(1, 2) - I).abs().max() < 1e-4)
show("mixing weights are SIGNED (cancellation possible)", bool((U < -1e-6).any()))

# ---- F3: forward pass, masks, normalization --------------------------
from training.evaluate import batch_to_model

inputs = batch_to_model(batch, device)
y = batch["y"].to(device)

with torch.no_grad():
    out = model(**inputs, y=y)

probs = out["scores"].exp()
mass = (probs * inputs["O_mask"]).sum(1)

show("scores are normalized log-probs",
     torch.allclose(mass, torch.ones_like(mass), atol=1e-3))
show("padded options get ~zero probability",
     float((probs * ~inputs["O_mask"]).max()) < 1e-6)

# ---- F4: are the complex amplitudes doing anything? ------------------
# This is the v45 claim. phase_effect ~ 0 means the phases are inert and
# the Born rule has degenerated back into a softmax over a classical
# mixture; destructive_fraction ~ 0 means nothing is cancelling, which no
# real non-negative amplitude scheme could do anyway.
show("phases move the result", float(out["phase_effect"]) > 1e-3,
     f"phase_effect {float(out['phase_effect']):.4f}")
show("destructive interference occurs", float(out["destructive_fraction"]) > 0,
     f"destructive_fraction {float(out['destructive_fraction']):.4f}")

# ---- F5: loss is finite and gradients flow ---------------------------
from training.losses import compute_loss

model.zero_grad()
loss = compute_loss(model(**inputs, y=y), y)
loss.backward()

grads = [p.grad for p in model.parameters() if p.grad is not None]
show("loss finite, gradients flow",
     torch.isfinite(loss) and all(torch.isfinite(g).all() for g in grads),
     f"loss {loss.item():.4f}, {len(grads)} tensors with grad")
model.zero_grad()

print("\n" + ("ALL SANITY CHECKS PASSED" if ok else "SOME CHECKS FAILED -- stop here"))
assert ok

<a id="10"></a>
## 10. Train

Interrupting is safe — a `_latest.pt` lands in Drive every epoch, so
re-running the trainer cell resumes from the last completed epoch.

**Watch the diagnostics, not the accuracy.** Four of them decide whether
anything below means what it says:

| metric | inert value | what it means |
|---|---|---|
| `Pairwise Cos` | → 1.0 | hypotheses produced identical energy rows; the multi-hypothesis mechanism is dead |
| `H Cos` | → 1.0 | the *reasoner* merged them into one vector (a different fix from the above) |
| `Phase effect` | → 0 | the phases learned nothing; the Born rule is a softmax again |
| `destructive` | → 0 | nothing is cancelling; a classical mixture reproduces this exactly |

The trainer prints an explicit `[COLLAPSE WARNING]` / `[INTERFERENCE
WARNING]` when these cross their thresholds. v44 reported a plausible
33.7% with `pairwise_cos` sitting at 1.000000.

In [ ]:
trainer = Trainer(
    model=model, train_loader=train_loader, val_loader=val_loader,
    device=device, ckpt_dir=CKPT_DIR, name=RUN_NAME,
    weight_decay=WEIGHT_DECAY, verbose=True,
)

start_epoch, best_acc, _ = load_or_resume(trainer, CKPT_DIR, RUN_NAME, EPOCHS)
print(f"\nstarting from epoch {start_epoch}/{EPOCHS}  (best so far {best_acc:.4f})")

In [ ]:
history = trainer.train(
    epochs=EPOCHS, start_epoch=start_epoch,
    best_acc=best_acc, patience=PATIENCE,
)

In [ ]:
torch.save(
    {
        "use_quantum": True, "use_validator": True,
        "persistent_steps": persistent_steps, "n_qubits": n_qubits,
        "use_question": True, "model_dim": MODEL_DIM,
        "phase_mode": PHASE_MODE, "phase_source": PHASE_SOURCE,
        "mixing": MIXING, "use_llm_prior": USE_LLM_PRIOR, "use_attack": True,
        "arc_config": ARC_CONFIG, "seed": SEED,
        "train_samples": TRAIN_SAMPLES, "val_samples": VAL_SAMPLES,
    },
    os.path.join(CKPT_DIR, f"{RUN_NAME}_config.pt"),
)
print("[CONFIG SAVED]")

### 10b. Training curves

At 80 training examples the interesting curve is not accuracy — it is
whether train loss falls while validation accuracy does not. That is the
signature of the capacity/data confound v44 ran into, and it makes every
mechanistic conclusion unsafe.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 4, figsize=(19, 3.8))

ax[0].plot(history["loss"]); ax[0].set_title("train loss")

ax[1].plot(history["acc"]); ax[1].axhline(0.25, ls="--", c="r", label="chance")
ax[1].set_title("val accuracy"); ax[1].legend()

ax[2].plot(history["pairwise_cos"], label="pairwise cos")
ax[2].plot(history["h_cos"], label="h cos")
ax[2].axhline(0.99, ls="--", c="r", label="collapse")
ax[2].set_ylim(0, 1.05); ax[2].set_title("collapse detectors"); ax[2].legend()

ax[3].plot(history["phase_effect"], label="phase effect")
ax[3].plot(history["destructive_fraction"], label="destructive frac")
ax[3].set_title("interference"); ax[3].legend()

for a in ax:
    a.set_xlabel("epoch")

plt.tight_layout()
plt.savefig(f"{EXPORT_DIR}/{RUN_NAME}_curves.png", dpi=120, bbox_inches="tight")
plt.show()

<a id="11"></a>
## 11. Evaluation

Reload the **best** checkpoint rather than evaluating whatever weights
the last epoch happened to leave behind.

In [ ]:
best_path = os.path.join(CKPT_DIR, f"{RUN_NAME}_best.pt")
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=device)["model"])
    print(f"[LOADED] {best_path}")

eval_model = model
metrics = evaluate(eval_model, val_loader, device)

for k, v in metrics.items():
    print(f"  {k:<22s}: {v:.4f}")

se = (metrics["acc"] * (1 - metrics["acc"]) / len(val_ds)) ** 0.5
print(f"\n  n = {len(val_ds)}, so the standard error is ~{se:.3f} "
      f"(+/-{100*se:.0f} points).")
print("  Any comparison smaller than that is noise.")

<a id="12"></a>
## 12. ⭐ Input ablation — which inputs is this model actually using?

**The most important cell in this notebook.**

qAIR claims to reason over multiple hypotheses held in superposition.
That claim is falsifiable in one line: corrupt the hypotheses and see
whether the answer changes. Pre-v44 it did not — not "changed a little",
but **bit-identically unchanged** at 294/869 under every corruption
tried, while only `O := zeros` moved the number.

v45 makes it per-channel, because there is now a second way to look good
for the wrong reason: the cached **LLM log-prior** is a strong feature
that can carry the accuracy on its own. A model that survives
`H := zeros` and collapses on `llm_logprob := zeros` is a wrapper around
its own generator, and reporting that as multi-hypothesis reasoning would
be the same category of error the v44 audit found, one level up.

(At n=20 this is a plumbing check. The deltas are ±5 points per example.)

In [ ]:
from evaluation.input_ablation import input_ablation, report

ab_results = input_ablation(eval_model, val_loader, device)
passed = report(ab_results)

<a id="13"></a>
## 13. Reference baselines

Two numbers that decide how to read everything above.

**Direct LLM** — the generator's own per-option log-likelihood, cached at
build time so it is free. This is the number that decides whether the
project has a result at all: if the pipeline cannot beat the model it is
built on top of, it is a lossy compression of that model's knowledge.
It went **unmeasured for the project's entire history**.

**Data ceiling** — probes straight on the cached embeddings. No
architecture over this cache can do much better, so a qAIR number below
these is a problem with qAIR, not with the task. On the v44 cache
`Q × O` scored 0.4849 and adding the hypotheses *dropped* it to 0.4408 —
the evidence channel was actively harmful. `Q × O + support` vs `Q × O`
is the same question for v45.

In [ ]:
from evaluation.baselines import run_cached_llm_baseline

llm_acc, llm_records = run_cached_llm_baseline(
    CACHE_DIR, split="validation", max_samples=VAL_SAMPLES,
    arc_config=ARC_CONFIG,
)

print(f"\nqAIR   : {metrics['acc']:.4f}")
print(f"direct : {llm_acc:.4f}")

In [ ]:
from evaluation.baselines import run_probes

probe_results = run_probes(
    CACHE_DIR, train_samples=TRAIN_SAMPLES, val_samples=VAL_SAMPLES,
    arc_config=ARC_CONFIG,
)

<a id="14"></a>
## 14. Ablation grid

The grid is deconfounded: each arm changes exactly one variable from its
nearest neighbour. Four contrasts carry the scientific weight:

| contrast | question |
|---|---|
| `A3` vs `P0` | does coherent summation beat the decohered mixture? |
| `A3` vs `P1` | do the **phases** contribute, or just the coherent sum? |
| `A1b` vs `A1c` | does the quantum circuit beat a parameter-matched classical MLP in the same slot? (54 of ~1.05M params, so a gap is not capacity) |
| `A3` vs `E0` | does the attacking-evidence channel earn its keep? |
| `F0` vs `A3` | how much of the system is just its own generator? |

Running a **subset at one seed** here. That is a plumbing check, not a
measurement: at n=20 nothing in this table is distinguishable from
anything else. The real sweep is `config.SEEDS` × the full grid on the
full split.

In [ ]:
from training.ablations import run_ablation_suite, ABLATIONS

RUN_GRID = True

GRID_CONFIGS = [
    "A1_baseline",
    "A3_persistent",
    "P0_classical_collapse",
    "F0_full_with_llm_prior",
]

print("full grid:")
for name in ABLATIONS:
    mark = "*" if name in GRID_CONFIGS else " "
    print(f" {mark} {name}")

ablation_results = None

if RUN_GRID:
    ablation_results = run_ablation_suite(
        cache_dir=CACHE_DIR, ckpt_dir=CKPT_DIR,
        epochs=EPOCHS, patience=PATIENCE, n_qubits=n_qubits,
        seeds=(SEED,), with_test=False, arc_config=ARC_CONFIG,
        configs=GRID_CONFIGS,
        train_samples=TRAIN_SAMPLES, val_samples=VAL_SAMPLES,
    )

<a id="15"></a>
## 15. Statistics

Bootstrap CIs plus **paired** McNemar tests. Paired matters: two arms see
the same examples in the same order, so testing the disagreement cells is
far more powerful than comparing two independent proportions.

With 4+ arms at α=0.05, apply Holm–Bonferroni before claiming any single
contrast in a write-up.

In [ ]:
from evaluation.stats import compare_arms, bootstrap_ci

if ablation_results:
    compare_arms(ablation_results, baseline="A1_baseline")
else:
    print("No grid results -- run 14 with RUN_GRID=True first.\n")

_, recs = evaluate(eval_model, val_loader, device, return_records=True)
acc, lo, hi = bootstrap_ci(recs)

print(f"\n{RUN_NAME}: acc={acc:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")
print(f"chance = 0.2500, direct LLM = {llm_acc:.4f}")

from evaluation.stats import mcnemar
b, c, p = mcnemar(llm_records, recs)
print(f"paired McNemar vs the direct LLM: b={b} c={c} p={p:.4f}")

<a id="16"></a>
## 16. Qualitative sample inference

The ARC numbers say *how often*; they say nothing about *how*. This runs
ten hand-written probes — plain science questions plus a couple of traps
(a rooster that cannot lay eggs) — through the live pipeline and prints
each option's support and attack side by side.

This is the cell that tells you whether the generator is producing
**evidence** or just fluent text, which no aggregate accuracy will
reveal. Read a few attacks on the *correct* option: if they are as
specific and convincing as the attacks on wrong options, the asymmetry
v45 depends on does not exist and §6b will already have said so.

In [ ]:
from evaluation.sample_inference import run_sample_evaluation

sample_acc = run_sample_evaluation(model=eval_model, device=device)

print(f"\nSample accuracy: {sample_acc:.4f}  (10 hand-written questions)")
print("n=10, so this is a smoke test of the live pipeline, not a metric.")

<a id="17"></a>
## 17. Summary

Reads the run back and states plainly what it does and does not support.

In [ ]:
print("=" * 72)
print(f"qAIR-vNext v46  --  {RUN_NAME}  ({ARC_CONFIG})")
print("=" * 72)
print(f"data           : {len(train_ds)} train / {len(val_ds)} val  (capped)")
print(f"backend        : {model.backend}   phase={model.phase_mode}/"
      f"{model.phase_source}   mixing={model.mixing}")
print(f"evidence       : {GEN_MODES}   llm_prior={model.use_llm_prior}")
print(f"parameters     : {total:,}  (circuit {circuit:,})")
print()
print(f"val accuracy   : {metrics['acc']:.4f}   [95% CI {lo:.4f}, {hi:.4f}]")
print(f"chance         : 0.2500")
print(f"direct LLM     : {llm_acc:.4f}")
print(f"ECE            : {metrics['ece']:.4f}")
print()
print("MECHANISM CHECKS")
print(f"  pairwise cos          : {metrics['pairwise_cos']:.4f}   "
      f"({'COLLAPSED' if metrics['pairwise_cos'] > 0.99 else 'ok'})")
print(f"  h cos                 : {metrics['h_cos']:.4f}")
print(f"  phase effect          : {metrics['phase_effect']:.4f}   "
      f"({'INERT' if metrics['phase_effect'] < 0.01 else 'active'})")
print(f"  destructive fraction  : {metrics['destructive_fraction']:.4f}   "
      f"({'no cancellation' if metrics['destructive_fraction'] < 0.01 else 'cancelling'})")
print(f"  input ablation        : {'PASS' if passed else 'FAIL'}")
print()
print("WHAT THIS RUN SUPPORTS")
print("  * the v45 pipeline executes end to end: cache -> train -> eval")
print("  * the mechanism checks above are live and would fire if broken")
print()
print("WHAT IT DOES NOT SUPPORT")
print(f"  * any accuracy claim. n={len(val_ds)}, SE ~{se:.3f}; the CI spans "
      f"{100*(hi-lo):.0f} points")
print(f"  * any ablation conclusion. One seed, {len(train_ds)} training examples")
print("  * any quantum claim beyond 'the machinery is not inert at this scale'")
print("=" * 72)

---

### Next steps

1. **Scale the cache up.** 80/20 answers "does it run", not "how well".
   ```bash
   python scripts/build_cache.py --splits train validation
   ```
   Full ARC-Challenge is 1,119 / 299 — about 15× this run, still under an
   hour on a T4.

2. **Build the test split.** Validation currently does double duty as
   both model selection and reported metric, which biases it upward:
   you are reporting the maximum over N epochs of a noisy quantity,
   measured on the data used to pick the maximum.
   ```bash
   python scripts/build_cache.py --splits test
   ```

3. **Run the full grid across `config.SEEDS`.** ARC-Challenge validation
   is 299 examples, SE ≈ 2.7 points; single-seed gaps below ~5 points are
   not interpretable.

4. **Then, and only then, read the four contrasts** in §14. If `A3` does
   not beat `P0`, the complex amplitudes are decoration and the honest
   move is to say so — that finding is publishable and the field needs
   it more than another architecture that reports a plausible number.